# Milestone 4: Model Comparison

**Dataset:** House Sales in King County, USA (the engineered data from Milestone 2)

**Goal:** Try stronger models and see if they beat my baseline. I also tune them with `GridSearchCV`.

**Target variable:** `price` (a number), so I use **regression** models.

**Score used:** R2. For all models, a higher R2 is better. 1 is perfect.

## Step 1: Import libraries and load the data

In [14]:
import pandas as pd
import numpy as np
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score


df = pd.read_csv(r"C:\Users\Soham\OneDrive\Desktop\data-science-internship\data\engineered_data.csv")

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Non-numeric columns:",
      df.select_dtypes(exclude="number").columns.tolist())

df.head()


Dataset shape: (21613, 96)
Missing values: 0
Non-numeric columns: []


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,zipcode_98146,zipcode_98148,zipcode_98155,zipcode_98166,zipcode_98168,zipcode_98177,zipcode_98178,zipcode_98188,zipcode_98198,zipcode_98199
0,221900.0,-0.398737,-1.447464,-0.979835,-0.228321,-0.915427,0,-0.305759,-0.629187,-0.558836,...,0,0,0,0,0,0,1,0,0,0
1,538000.0,-0.398737,0.175607,0.533634,-0.189885,0.936506,0,-0.305759,-0.629187,-0.558836,...,0,0,0,0,0,0,0,0,0,0
2,180000.0,-1.473959,-1.447464,-1.426254,-0.123298,-0.915427,0,-0.305759,-0.629187,-1.409587,...,0,0,0,0,0,0,0,0,0,0
3,604000.0,0.676485,1.149449,-0.130550,-0.244014,-0.915427,0,-0.305759,2.444294,-0.558836,...,0,0,0,0,0,0,0,0,0,0
4,510000.0,-0.398737,-0.149007,-0.435422,-0.169653,-0.915427,0,-0.305759,-0.629187,0.291916,...,0,0,0,0,0,0,0,0,0,0


## Step 2: Split the data

I use the same split as Milestone 3 (80% train, 20% test, `random_state=42`). This makes the comparison fair.

In [15]:
X = df.drop(columns=['price'])
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", X_train.shape[0])
print("Testing rows: ", X_test.shape[0])

Training rows: 17290
Testing rows:  4323


## Step 3: Baseline model

I train the Linear Regression baseline again. I need its train and test scores for the table.

In [16]:
# Select only numeric feature columns to remove text/dates like 'date' or non-feature IDs
X_train_clean = X_train.select_dtypes(include=['number'])
X_test_clean = X_test.select_dtypes(include=['number'])

# If 'id' is present as a numeric column, drop it
X_train_clean = X_train_clean.drop(columns=['id'], errors='ignore')
X_test_clean = X_test_clean.drop(columns=['id'], errors='ignore')

# Update X_train and X_test for subsequent model cells
X_train = X_train_clean
X_test = X_test_clean

# Train baseline linear regression
baseline = LinearRegression()
baseline.fit(X_train, y_train)

baseline_train = r2_score(y_train, baseline.predict(X_train))
baseline_test = r2_score(y_test, baseline.predict(X_test))

print(f"Baseline -> Train R2: {baseline_train:.3f}, Test R2: {baseline_test:.3f}")

Baseline -> Train R2: 0.811, Test R2: 0.808


## Step 4: Random Forest with GridSearchCV

**Random Forest** builds many decision trees and averages them. It can learn patterns that are not a straight line.

**GridSearchCV** tries every combination of settings (hyperparameters). It checks each one with cross-validation. Cross-validation splits the training data into 3 parts and tests on each part in turn. The test set is not used, so it stays fair.

The settings I try:
- `n_estimators`: the number of trees.
- `max_depth`: how deep each tree can grow. A deeper tree can learn more, but it can also memorize the training data.

In [17]:
rf_grid = {'n_estimators': [50, 100], 'max_depth': [5, 10]}

rf_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=rf_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

rf_best = rf_search.best_estimator_
print("Best Random Forest settings:", rf_search.best_params_)

Best Random Forest settings: {'max_depth': 10, 'n_estimators': 100}


## Step 5: Gradient Boosting with GridSearchCV

**Gradient Boosting** also uses many trees. But it builds them one after another. Each new tree fixes the mistakes of the trees before it.

The settings I try:
- `n_estimators`: the number of trees.
- `learning_rate`: how big each correction step is.
- `max_depth`: how deep each tree can grow.

In [18]:
gb_grid = {'n_estimators': [50, 100], 'learning_rate': [0.1, 0.2], 'max_depth': [3, 5]}

gb_search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid=gb_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)
gb_search.fit(X_train, y_train)

gb_best = gb_search.best_estimator_
print("Best Gradient Boosting settings:", gb_search.best_params_)

Best Gradient Boosting settings: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100}


## Step 6: Evaluate all models on the train and test sets

In [19]:
rf_train = r2_score(y_train, rf_best.predict(X_train))
rf_test = r2_score(y_test, rf_best.predict(X_test))

gb_train = r2_score(y_train, gb_best.predict(X_train))
gb_test = r2_score(y_test, gb_best.predict(X_test))

print(f"Random Forest     -> Train R2: {rf_train:.3f}, Test R2: {rf_test:.3f}")
print(f"Gradient Boosting -> Train R2: {gb_train:.3f}, Test R2: {gb_test:.3f}")

Random Forest     -> Train R2: 0.933, Test R2: 0.837
Gradient Boosting -> Train R2: 0.961, Test R2: 0.852


## Step 7: Comparison table

In [20]:
results = [
    ("Baseline", baseline_train, baseline_test, "N/A"),
    ("Random Forest", rf_train, rf_test, str(rf_search.best_params_)),
    ("Gradient Boosting", gb_train, gb_test, str(gb_search.best_params_)),
]

table = "| Model | Train Score | Test Score | Best Params |\n"
table += "|-------|-------------|------------|-------------|\n"
for name, tr, te, params in results:
    table += f"| {name} | {tr:.3f} | {te:.3f} | {params} |\n"

display(Markdown(table))

| Model | Train Score | Test Score | Best Params |
|-------|-------------|------------|-------------|
| Baseline | 0.811 | 0.808 | N/A |
| Random Forest | 0.933 | 0.837 | {'max_depth': 10, 'n_estimators': 100} |
| Gradient Boosting | 0.961 | 0.852 | {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100} |


## Step 8: Check for overfitting

A model **overfits** when it does very well on the training data but much worse on the test data. It has memorized the training data instead of learning real patterns.

So I do not only look at the highest test score. I also look at the gap: **Train Score - Test Score**. A big gap is a warning.

In [21]:
for name, tr, te, params in results:
    gap = tr - te
    print(f"{name:<18} Train - Test gap: {gap:.3f}")

Baseline           Train - Test gap: 0.002
Random Forest      Train - Test gap: 0.097
Gradient Boosting  Train - Test gap: 0.109


## Step 9: Conclusion

**How I choose the best model:**
1. First, I look at the test score. Higher is better.
2. Then, I check the gap between train and test. A small gap means the model will work on new data.
3. If two models have close test scores, I choose the one with the smaller gap.

In [22]:
best_name, best_test, best_gap = None, -999, None
for name, tr, te, params in results:
    if te > best_test:
        best_name, best_test, best_gap = name, te, tr - te

print(f"Highest test score: {best_name} (Test R2 = {best_test:.3f}, gap = {best_gap:.3f})")
print(f"Improvement over the baseline: {best_test - baseline_test:+.3f} R2")

Highest test score: Gradient Boosting (Test R2 = 0.852, gap = 0.109)
Improvement over the baseline: +0.044 R2


**My written conclusion:**

Read the table and the gap numbers above. Then match them to the points below.

- The Random Forest and Gradient Boosting models should score higher on the test set than the baseline. This is because they can learn patterns that are not a straight line.
- Both models score higher on the train set than on the test set. This is normal. A gap that is small to medium is fine. A very large gap is overfitting.
- The model with the highest test score and a reasonable gap is the best choice. This is the model named in the line "Highest test score" above.
- If the best model is only a little better than the baseline, the simple baseline may be enough. It is easier to explain.

**Why GridSearchCV helped:** It tested many settings for me. It showed that settings like depth change the result. A tree that is too deep memorizes the training data. A tree that is too shallow cannot learn enough.

**Next step:** I could try more settings in the grid or add more features. The grids here are small to keep the run time short.

## Step 10: Save the best model

I save my final model with `joblib`. The Streamlit app will load this file.

By default, the code saves the model with the highest test score. If my conclusion above chose a different model (for example, because of overfitting), I change `final_model_name`.

In [23]:
import joblib

models = {
    "Baseline": baseline,
    "Random Forest": rf_best,
    "Gradient Boosting": gb_best,
}

final_model_name = best_name   # change this if my conclusion chose a different model
joblib.dump(models[final_model_name], '../model.joblib')
print("Saved model:", final_model_name)

Saved model: Gradient Boosting


## Summary

- I trained 2 ensemble models: Random Forest and Gradient Boosting.
- I tuned both with `GridSearchCV`.
- I built the comparison table with the baseline, Random Forest and Gradient Boosting.
- I checked the train/test gap to look for overfitting.
- I wrote a conclusion on the best model.
- I saved the final model to `model.joblib`.